In [ ]:
from pyspark.sql import SparkSession
# 1. Khởi tạo Spark Session kết nối đến Master chung của cụm
spark = SparkSession.builder \
    .appName("MetroPT3_Machine_Learning") \
    .master("spark://10.125.222.18:7077") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()
# 2. Đường dẫn nạp dữ liệu sạch đã lưu trên HDFS
HDFS_ML = "hdfs://10.125.222.18:9000/user/bigdata/cleaned/metropt3_clean_for_sql"
df = spark.read.parquet(HDFS_ML)
# 4. Kiểm tra cấu trúc dữ liệu để bắt đầu làm Classification
df.printSchema()
print(f"Tổng số bản ghi: {df.count()}")
df.createOrReplaceTempView('sensor')

In [ ]:
# Q9: PHÂN TÍCH CÁC PHIÊN VẬN HÀNH LIÊN TỤC DÀI NHẤT
q9 = spark.sql('''
WITH session_edges AS (
    SELECT
        timestamp,
        COMP,
        Motor_current,
        LAG(COMP,1,1) OVER(ORDER BY timestamp) AS prev_comp FROM sensor),
session_blocks AS (
    SELECT
        timestamp,
        COMP,
        Motor_current,
        SUM(
            CASE
                WHEN COMP = 0 AND prev_comp = 1 THEN 1
                ELSE 0 END
        ) OVER(ORDER BY timestamp) AS session_id
    FROM session_edges)
SELECT
    session_id,
    MIN(timestamp) AS session_start,
    MAX(timestamp) AS session_end,
    COUNT(*) AS duration_seconds,
    ROUND(AVG(Motor_current),2) AS avg_current_in_session
FROM session_blocks
WHERE COMP = 0
GROUP BY session_id
ORDER BY duration_seconds DESC
LIMIT 10''')
print("\n========== EXECUTION PLAN ==========")
q9.explain(True)
print("\n--- TOP 10 PHIÊN VẬN HÀNH LIÊN TỤC DÀI NHẤT ---")
df_q9 = q9.toPandas()
try:
    display(df_q9)
except NameError:
    print(df_q9.to_string(index=False))